In [33]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import numpy as np
from PIL import Image

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [35]:
def get_random_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count == 0:
        return None
    rand_index = np.random.randint(0, frame_count)
    cap.set(cv2.CAP_PROP_POS_FRAMES, rand_index)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

In [36]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [42]:
class DeepFakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.video_paths = []
        self.labels = []
        self.transform = transform

        for label, subdir in enumerate(['videos_real', 'videos_fake']):
            folder = os.path.join(root_dir, subdir)
            for filename in os.listdir(folder):
                if filename.endswith(".mp4"):
                    self.video_paths.append(os.path.join(folder, filename))
                    self.labels.append(label)

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frame = get_random_frame(video_path)
        while frame is None:  # skip unreadable videos
            idx = (idx + 1) % len(self.video_paths)
            video_path = self.video_paths[idx]
            label = self.labels[idx]
            frame = get_random_frame(video_path)

        if self.transform:
            frame = self.transform(frame)

        return frame, torch.tensor(label, dtype=torch.float)

In [43]:
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.model.fc = nn.Linear(self.model.fc.in_features, 1)

    def forward(self, x):
        return self.model(x).squeeze(1)

In [44]:
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [40]:
def evaluate(model, dataloader, device):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            predicted = torch.sigmoid(outputs).cpu().numpy()
            preds.extend((predicted > 0.5).astype(int))
            true_labels.extend(labels.numpy())

    return preds, true_labels

In [45]:
dataset_path = r"C:\Users\rimjh\Downloads\dataset"
dataset = DeepFakeDataset(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [46]:
model = CNNClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

In [47]:
for epoch in range(5):
    loss = train(model, dataloader, optimizer, criterion, device)
    preds, labels = evaluate(model, dataloader, device)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    print(f"Epoch {epoch+1} - Loss: {loss:.4f} | Acc: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

100%|██████████| 14/14 [00:08<00:00,  1.65it/s]


Epoch 1 - Loss: 0.8682 | Acc: 0.6226 | Precision: 0.6000 | Recall: 0.7358 | F1: 0.6610


100%|██████████| 14/14 [00:08<00:00,  1.73it/s]


Epoch 2 - Loss: 0.7428 | Acc: 0.7075 | Precision: 0.7115 | Recall: 0.6981 | F1: 0.7048


100%|██████████| 14/14 [00:08<00:00,  1.64it/s]


Epoch 3 - Loss: 0.6452 | Acc: 0.6415 | Precision: 0.8261 | Recall: 0.3585 | F1: 0.5000


100%|██████████| 14/14 [00:08<00:00,  1.62it/s]


Epoch 4 - Loss: 0.6685 | Acc: 0.8113 | Precision: 0.8511 | Recall: 0.7547 | F1: 0.8000


100%|██████████| 14/14 [00:08<00:00,  1.61it/s]


Epoch 5 - Loss: 0.5766 | Acc: 0.8208 | Precision: 0.7742 | Recall: 0.9057 | F1: 0.8348
